# Data cleaning pipeline

This notebook builds MatchCast's processed match table from the raw
`martj42/international_results` snapshot. It is built up section by section,
one cleaning concern at a time, matching the schema defined in
`01_processed_match_schema.ipynb`. Each section documents its rule and
validates it before the next section runs.

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/international_results.csv")
PROCESSED_PATH = Path("../data/processed/matches.csv")

raw = pd.read_csv(RAW_PATH, dtype="string", keep_default_na=False)
raw["source_row_number"] = range(len(raw))

print(f"Loaded {len(raw)} raw rows from {RAW_PATH}")

Loaded 49501 raw rows from ..\data\raw\international_results.csv


## Parse and sort match dates

Dates are parsed with an explicit `YYYY-MM-DD` format so malformed values
surface as `NaT` instead of being silently misparsed. Rows are then sorted by
`date` using a stable sort with `source_row_number` as the deterministic
tiebreaker, so matches played on the same day keep their original upstream
order every time this notebook runs.

In [2]:
matches = raw.copy()
matches["date"] = pd.to_datetime(
    matches["date"].str.strip(), format="%Y-%m-%d", errors="raise"
)
matches = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)

print(f"Date range: {matches['date'].min().date()} to {matches['date'].max().date()}")

Date range: 1872-11-30 to 2026-07-06


### Validation: date parsing and sort determinism

In [3]:
assert matches["date"].notna().all(), "every row must have a parsed date"
assert len(matches) == len(raw), "sorting must not drop or add rows"
assert matches["date"].is_monotonic_increasing, "rows must be sorted by date"

resorted = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)
assert matches["source_row_number"].tolist() == resorted["source_row_number"].tolist(), (
    "re-sorting the already-sorted table must be a no-op (deterministic order)"
)

same_day = matches[matches.duplicated(subset="date", keep=False)]
for _, group in same_day.groupby("date"):
    assert group["source_row_number"].is_monotonic_increasing, (
        "same-day matches must preserve original upstream order"
    )

print("Date parsing and deterministic sort validated.")

Date parsing and deterministic sort validated.


## Normalize team names and preserve originals

Team names are normalized (surrounding whitespace stripped, internal
whitespace collapsed) and passed through an explicit alias mapping before use
in any downstream feature or model. The unnormalized values are preserved in
`original_home_team` / `original_away_team` so every processed row remains
traceable to the exact upstream string, even if a future refresh or a
supplementary source (see `docs/data-sources.md`) introduces a name this
mapping does not yet cover.

The current snapshot's team names are already whitespace-clean and
single-form (verified below), so `TEAM_NAME_ALIASES` starts empty. The
mapping exists so a future snapshot revealing spelling variants (for example
an alternate name for the same team) can be corrected in one place without
rewriting the pipeline.

In [4]:
TEAM_NAME_ALIASES: dict[str, str] = {}


def normalize_team_name(name: str, aliases: dict[str, str] = TEAM_NAME_ALIASES) -> str:
    """Collapse whitespace and apply the explicit alias mapping."""
    normalized = " ".join(name.strip().split())
    return aliases.get(normalized, normalized)


matches["original_home_team"] = matches["home_team"]
matches["original_away_team"] = matches["away_team"]
matches["home_team"] = matches["home_team"].map(normalize_team_name)
matches["away_team"] = matches["away_team"].map(normalize_team_name)

print(f"Distinct normalized teams: {pd.unique(matches[['home_team', 'away_team']].values.ravel()).size}")

Distinct normalized teams: 336


### Validation: normalization and traceability

In [5]:
for column in ("home_team", "away_team"):
    assert (matches[column] == matches[column].str.strip()).all(), (
        f"{column} must not have leading/trailing whitespace"
    )
    assert (~matches[column].str.contains(r"\s{2,}", regex=True)).all(), (
        f"{column} must not contain repeated internal whitespace"
    )
    assert (matches[column] != "").all(), f"{column} must not be empty after normalization"

# Normalization must be idempotent.
assert matches["home_team"].map(normalize_team_name).equals(matches["home_team"])
assert matches["away_team"].map(normalize_team_name).equals(matches["away_team"])

# Original values must be preserved untouched for traceability.
assert matches["original_home_team"].equals(raw.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)["home_team"])
assert matches["original_away_team"].equals(raw.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)["away_team"])

print("Team name normalization and traceability validated.")

Team name normalization and traceability validated.
